Starting with $u(x,y)=1-3x^2+2x^3$ again.  
Using the $-\Delta u = f$ and hard-coding the $f$.  
$-\Delta(u) = -u_{xx}-u_{yy}=-(-6+12x)-0=6-12x$  

Result: Only solved using CG, seems to work

<span style="color:red">Current Concern: Calculating actual errors to justify success</span>



In [ ]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Domain is unit square, create initial mesh
N = 64 
mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
mesh.nv, mesh.ne
Draw(mesh);

In [ ]:
# Working with H1-conforming piecewise linear finite elements
# boundary conditions are Dirichlet on the left and default=Neumann on the other 3 sides
fes = H1(mesh, order=1, dirichlet="left")
fes.ndof

# Solution will be stored as a grid function
gfu = GridFunction(fes)
gfu.Set(1, BND) # this sets the Dirichlet boundary component to 1, might want to modify later
Draw(gfu)

In [ ]:
# Set test and trial spaces
u = fes.TrialFunction()
phi = fes.TestFunction()

# Bilinear form
a = BilinearForm(fes)
a += grad(u)*grad(phi)*dx # Laplacian
a.Assemble()

print(a.mat)

In [ ]:
# Right hand side
f = LinearForm(fes)
f += (6 - 12 * x)*phi*dx # manually calculated 
f.Assemble()

print(f.vec)

In [ ]:
# Solve the PDE using CG
c = Preconditioner(a,"local")
c.Update()
solvers.BVP(bf=a, lf=f, gf=gfu, pre=c, maxsteps=500, tol=1e-14)
Draw(gfu)

In [ ]:
# Plot the original function for comparison
plant = CoefficientFunction(1-3*x*x+2*x*x*x)
Draw(plant, mesh)